# TruthLens Video Model v2: fine-tune Video Model v1 on RTFS + FakeAVCeleb

Fine-tunes the deployed **Video Model v1** (EfficientNet-B0, `epoch_11_model_only.pt`) on production-identical face
crops, with real-world degradation augmentation (JPEG re-compression, downscaling, blur, noise).

**Fully resumable.** Everything that matters lives on Google Drive under `MyDrive/TruthLens_video_ft/`:
- `crops/`: the 16 face crops per video, extracted once with the app's own code (cloned from GitHub at a pinned commit)
- `run/latest.pt`: model + optimiser + scheduler + position, saved every 400 steps and at every epoch end
- `run/best.pt`: best epoch by validation AUC (this is the file to bring back to the Mac)
- `run/history.json`: per-epoch metrics

If Colab disconnects or the GPU quota runs out, reconnect and **Run all** again. Finished work is skipped and training
continues from the last save.

**Before the first run**, put these in `MyDrive/TruthLens_video_ft/inputs/`:
1. `manifest.csv` (from `video_retrain/manifest.csv` in the repo)
2. `epoch_11_model_only.pt` (the deployed video checkpoint, `backend/checkpoints/video/`)
3. `kaggle_token.txt`: your Kaggle API token (Kaggle -> Settings -> API -> Create New Token), single line

**Data splits** (`manifest.csv`, no source clip or person in more than one split):
- train: RTFS 1,516 real + 1,516 fake (inswapper/uniface), FakeAVCeleb 395 real + 395 fake (FSGAN/Faceswap/Wav2Lip mixes)
- val: 159 videos (RTFS + FakeAVCeleb), used only to pick the best epoch
- test: 221 videos (RTFS + FakeAVCeleb + 60 Celeb-DF), reported for information only. The authoritative before/after
  test is re-run on the Mac with the production pipeline.

In [ ]:
# 1. Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, shutil
WORK = '/content/drive/MyDrive/TruthLens_video_ft'
for d in ('inputs', 'crops', 'run'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout if shutil.which('nvidia-smi') else 'WARNING: no NVIDIA GPU visible - training would run on CPU')

In [ ]:
# 2. App code at a pinned commit (the exact production frame sampling + Haar face crop), and inputs check
REPO_COMMIT = 'd360475'
if not os.path.isdir('/content/TruthLens'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/rudra2024-sys/TruthLens.git', '/content/TruthLens'], check=True)
subprocess.run(['git', '-C', '/content/TruthLens', 'checkout', '-q', REPO_COMMIT], check=True)
import sys
sys.path.insert(0, '/content/TruthLens/backend')
from app.services.video.model_v1 import common, optimized

import hashlib
CKPT = f'{WORK}/inputs/epoch_11_model_only.pt'
EXPECTED = '91eefd295e14a527cddcd3ac51e4c134341d95b9c249ccc721cb26691ffae94d'   # models/CHECKSUMS.sha256
assert os.path.isfile(f'{WORK}/inputs/manifest_r2.csv'), 'upload manifest_r2.csv to inputs/'
RUN_NAME = 'run_r2'
assert os.path.isfile(CKPT), 'upload epoch_11_model_only.pt to inputs/'
assert hashlib.sha256(open(CKPT, 'rb').read()).hexdigest() == EXPECTED, 'checkpoint checksum mismatch'
import csv
ROWS = list(csv.DictReader(open(f'{WORK}/inputs/manifest_r2.csv')))
print(len(ROWS), 'manifest rows; checkpoint OK')

In [ ]:
# 3. Kaggle datasets: only downloaded if some crops are still missing (a fresh VM loses /content/data)
def crop_path(r):
    return f"{WORK}/crops/{r['key']}.png"

missing = [r for r in ROWS if not os.path.exists(crop_path(r))]
print(len(missing), 'videos still need crops')
if missing:
    os.environ['KAGGLE_API_TOKEN'] = open(f'{WORK}/inputs/kaggle_token.txt').read().strip()
    subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)
    need = {r['dataset'] for r in missing}
    if 'rtfs' in need and not os.path.isdir('/content/data/rtfs/original_videos'):
        subprocess.run(['kaggle', 'datasets', 'download', 'stplusplus/rtfs-10k', '-p', '/content/data/rtfs', '--unzip', '-q'], check=True)
    if 'fakeavceleb' in need and not os.path.isdir('/content/data/fakeavceleb'):
        subprocess.run(['kaggle', 'datasets', 'download', 'shreyaty08/fakeavceleb', '-p', '/content/data/fakeavceleb', '--unzip', '-q'], check=True)
    if 'celebdf' in need:   # only 60 test files: fetch individually instead of the whole 10 GB dataset
        import time
        for r in [r for r in missing if r['dataset'] == 'celebdf']:
            dst = f"/content/data/celebdf/{r['path']}"
            if os.path.exists(dst):
                continue
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            for attempt in range(6):
                subprocess.run(['kaggle', 'datasets', 'download', 'reubensuju/celeb-df-v2', '-f', r['path'], '-p', os.path.dirname(dst), '-q'])
                z = dst + '.zip'
                if os.path.exists(z):
                    subprocess.run(['unzip', '-o', '-q', z, '-d', os.path.dirname(dst)]); os.remove(z)
                if os.path.exists(dst):
                    break
                time.sleep(5 * (attempt + 1) ** 2)   # Kaggle 429 back-off
    subprocess.run(['df', '-h', '/content'])

In [ ]:
# 4. Production-identical face crops -> Drive (resumable; skips anything already extracted)
import glob
import numpy as np
from PIL import Image
from multiprocessing import Pool

if 'dataset_root' not in globals():
    def dataset_root(name):
        marker = {'rtfs': 'original_videos', 'fakeavceleb': 'RealVideo-RealAudio', 'celebdf': 'Celeb-real'}[name]
        hits = glob.glob(f'/content/data/{name}/**/{marker}', recursive=True)
        return os.path.dirname(hits[0]) if hits else f'/content/data/{name}'

if 'ROOTS' not in globals():
    ROOTS = {d: dataset_root(d) for d in {r['dataset'] for r in missing}} if missing else {}
print(ROOTS)

import cv2

def _worker_init():
    cv2.setNumThreads(1)          # avoid OpenCV thread-pool deadlocks in forked workers

def extract(r):
    out = crop_path(r)
    if os.path.exists(out):
        return 'skip'
    src = os.path.join(ROOTS[r['dataset']], r['path'])
    try:
        cascade = common.get_face_cascade()
        frames = optimized._decode_selected_frames(src)
        crops, faces = [], 0
        for f in frames:
            crop_rgb, info = common.crop_face_or_center_with_info(f, cascade)
            crops.append(np.asarray(Image.fromarray(crop_rgb).resize((224, 224), Image.Resampling.BILINEAR), dtype=np.uint8))
            faces += info['face_bbox'] is not None
        os.makedirs(os.path.dirname(out), exist_ok=True)
        tmp = out + '.tmp.png'
        Image.fromarray(np.concatenate(crops, axis=1)).save(tmp)
        os.replace(tmp, out)            # atomic: a half-written file is never mistaken for a finished one
        return f'ok faces={faces}'
    except Exception as e:
        return f'ERR {type(e).__name__}: {e}'

if missing:
    from collections import Counter
    stats = Counter()
    with Pool(os.cpu_count(), initializer=_worker_init) as pool:
        for i, res in enumerate(pool.imap_unordered(extract, missing, chunksize=4)):
            stats[res.split(' ')[0]] += 1
            if i % 200 == 0:
                print(i, '/', len(missing), dict(stats), flush=True)
    print('done', dict(stats))
errs = [r['key'] for r in ROWS if not os.path.exists(crop_path(r))]
print(len(errs), 'videos without crops (unreadable/too short; excluded from training/eval)')

In [ ]:
# 5. Copy crops to local disk for fast training reads
LOCAL = '/content/crops_local'
subprocess.run(['rsync', '-a', f'{WORK}/crops/', f'{LOCAL}/'], check=True)
print(subprocess.run(['du', '-sh', LOCAL], capture_output=True, text=True).stdout)

In [ ]:
# 6. Resumable fine-tuning
import io, json, random, time
import torch, torch.nn as nn
from PIL import ImageFilter
from torch.utils.data import DataLoader, Dataset

# Round 2 (round 1: 8 epochs, lr 1e-4, no smoothing, no per-dataset cap -> near-perfect on its two training datasets,
# but extreme logits that flagged most unfamiliar real video as FAKE). Smoothing + caps + lower lr + fewer epochs
# aim at a less over-confident model; Celeb-DF real footage is now in training and validation.
EPOCHS, LR, BATCH, FRAMES_PER_VIDEO, SAVE_EVERY = 4, 5e-5, 64, 8, 400
LABEL_SMOOTHING = 0.1      # targets 0.05 / 0.95
CAP_PER_CLASS = 800        # per dataset per label per epoch, so RTFS (1,516 per class) can't dominate
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = torch.tensor(common.IMAGENET_MEAN).view(3, 1, 1); STD = torch.tensor(common.IMAGENET_STD).view(3, 1, 1)
N = common.NUM_FRAMES

def items(split):
    out = []
    for r in ROWS:
        p = f"{LOCAL}/{r['key']}.png"
        if r['split'] == split and os.path.exists(p):
            out.append((p, int(r['label']), r['dataset']))
    return out

TRAIN, VAL, TEST = items('train'), items('val'), items('test')
print(len(TRAIN), 'train /', len(VAL), 'val /', len(TEST), 'test videos')

def load_strip(p):
    a = np.asarray(Image.open(p).convert('RGB'))
    return [a[:, i * 224:(i + 1) * 224] for i in range(N)]

def degrade(img, rnd):
    # applied identically to real and fake, so "looks degraded" can't become a label shortcut
    if rnd.random() < 0.5: img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if rnd.random() < 0.35:
        s = rnd.uniform(0.25, 0.75); small = img.resize((max(16, int(224 * s)),) * 2, Image.Resampling.BILINEAR)
        img = small.resize((224, 224), Image.Resampling.BILINEAR)
    if rnd.random() < 0.25: img = img.filter(ImageFilter.GaussianBlur(rnd.uniform(0.3, 1.8)))
    if rnd.random() < 0.25:
        a = np.asarray(img, dtype=np.float32) + np.random.normal(0, rnd.uniform(2, 10), (224, 224, 3))
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if rnd.random() < 0.6:
        buf = io.BytesIO(); img.save(buf, 'JPEG', quality=rnd.randint(15, 90)); buf.seek(0); img = Image.open(buf).convert('RGB')
    return img

def to_tensor(a):
    t = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float() / 255.0
    return (t - MEAN) / STD

class Frames(Dataset):
    def __init__(self, frames): self.frames = frames
    def __len__(self): return len(self.frames)
    def __getitem__(self, k):
        p, i, y = self.frames[k]
        a = load_strip(p)[i]
        a = np.asarray(degrade(Image.fromarray(a), random.Random()))
        return to_tensor(a), torch.tensor(float(y))

def epoch_frames(ep):
    rnd = random.Random(1000 + ep)                     # deterministic per epoch -> exact mid-epoch resume
    vids = []
    for ds in sorted({d for *_, d in TRAIN}):
        for lab in (0, 1):
            pool = [t for t in TRAIN if t[2] == ds and t[1] == lab]
            vids += rnd.sample(pool, min(CAP_PER_CLASS, len(pool)))
    fr = [(p, i, y) for p, y, _ in vids for i in rnd.sample(range(N), FRAMES_PER_VIDEO)]
    rnd.shuffle(fr)
    return fr

def auc(s, y):
    s, y = np.asarray(s), np.asarray(y); pos, neg = s[y == 1], s[y == 0]
    return float(np.mean((pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :])))

@torch.no_grad()
def evaluate(model, its):
    model.eval(); sc = []
    for p, y, ds in its:
        x = torch.stack([to_tensor(a) for a in load_strip(p)]).to(device)
        sc.append((float(model(x).squeeze(-1).mean()), y, ds))      # video score = mean of 16 frame logits (production)
    res = {ds: auc([z for z, _, d in sc if d == ds], [y for _, y, d in sc if d == ds]) for ds in sorted({d for *_, d in sc})}
    res['mean'] = float(np.mean(list(res.values())))
    return res

model = common.build_model().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps_per_epoch = (len(epoch_frames(1)) + BATCH - 1) // BATCH
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps_per_epoch * EPOCHS, pct_start=0.15)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
RUN = f'{WORK}/{RUN_NAME}'
os.makedirs(RUN, exist_ok=True)

if os.path.exists(f'{RUN}/latest.pt'):
    st = torch.load(f'{RUN}/latest.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(st['model_state_dict']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
    scaler.load_state_dict(st['scaler']); start_ep, start_step, history, best = st['epoch'], st['step'], st['history'], st['best']
    print(f'resumed at epoch {start_ep} step {start_step}, best val {best:.4f}')
else:
    common.load_checkpoint(model, CKPT); model.to(device)
    base = evaluate(model, VAL); history = [{'epoch': 0, **base}]; best = base['mean']
    torch.save({'model_state_dict': model.state_dict(), 'epoch': 0, 'val': base}, f'{RUN}/best.pt')
    start_ep, start_step = 1, 0
    print('epoch 0 (deployed model) val AUC', base)

def save_latest(ep, step):
    tmp = f'{RUN}/latest.pt.tmp'
    torch.save({'model_state_dict': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': ep, 'step': step, 'history': history, 'best': best}, tmp)
    os.replace(tmp, f'{RUN}/latest.pt')

loss_fn = nn.BCEWithLogitsLoss()
for ep in range(start_ep, EPOCHS + 1):
    frames = epoch_frames(ep)[start_step * BATCH:]
    dl = DataLoader(Frames(frames), batch_size=BATCH, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)
    model.train(); t0 = time.time(); tot, n = 0.0, 0
    for k, (x, y) in enumerate(dl, start=start_step):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device.type == 'cuda'):
            loss = loss_fn(model(x).squeeze(-1), y * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / 2)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); n += 1
        if k % 100 == 0:
            print(f'ep{ep} step {k}/{steps_per_epoch} loss {tot / n:.4f} ({time.time() - t0:.0f}s)', flush=True)
        if (k + 1) % SAVE_EVERY == 0:
            save_latest(ep, k + 1)
    res = evaluate(model, VAL)
    history.append({'epoch': ep, 'train_loss': tot / max(n, 1), **res})
    print(f'== epoch {ep} val AUC {res}', flush=True)
    if res['mean'] > best:
        best = res['mean']
        torch.save({'model_state_dict': model.state_dict(), 'epoch': ep, 'val': res}, f'{RUN}/best.pt')
        print(f'   new best {best:.4f} -> best.pt', flush=True)
    start_step = 0
    save_latest(ep + 1, 0)
    json.dump(history, open(f'{RUN}/history.json', 'w'), indent=1)
print('training finished; best mean val AUC', best)

In [ ]:
# 7. Report (informational): deployed vs best on val and test. The authoritative test is re-run on the Mac.
deployed = common.build_model(); common.load_checkpoint(deployed, CKPT); deployed.to(device)
tuned = common.build_model(); tuned.load_state_dict(torch.load(f'{RUN}/best.pt', map_location='cpu', weights_only=False)['model_state_dict']); tuned.to(device)
for name, its in (('val', VAL), ('test', TEST)):
    print(name, 'deployed', evaluate(deployed, its))
    print(name, 'tuned   ', evaluate(tuned, its))
print(json.dumps(history, indent=1))

## When it's done

Download `MyDrive/TruthLens_video_ft/run/best.pt` and `history.json` and give them to Claude on the Mac. They get
re-tested there against the deployed model with the exact production pipeline before anything changes in the app.